In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ab_data.csv to ab_data.csv


In [ ]:
import pandas as pd
import sqlite3

In [ ]:
df = pd.read_csv("ab_data.csv")
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [ ]:
conn = sqlite3.connect("ab_test.db")
df.to_sql("ab_data", conn, if_exists="replace", index=False)

294478

In [ ]:
pd.read_sql("SELECT * FROM ab_data LIMIT 5;", conn)

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [ ]:
df = df.drop_duplicates(subset="user_id")

In [ ]:
df = df[((df["group"] == "control") & (df["landing_page"] == "old_page")) |
        ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))]

In [ ]:
df.shape

(288540, 5)

In [ ]:
df.to_sql("ab_data", conn, if_exists="replace", index=False)

288540

In [ ]:
pd.read_sql("""
SELECT "group", COUNT(*) AS total_users
FROM ab_data
GROUP BY "group";
""", conn)

,group,total_users
0,control,144226
1,treatment,144314


In [ ]:
pd.read_sql("""
SELECT "group", SUM(converted) AS total_conversions
FROM ab_data
GROUP BY "group";
""", conn)

,group,total_conversions
0,control,17349
1,treatment,17134


In [ ]:
pd.read_sql("""
SELECT
    "group",
    COUNT(*) AS total_users,
    SUM(converted) AS conversions,
    ROUND(1.0 * SUM(converted) / COUNT(*), 4) AS conversion_rate
FROM ab_data
GROUP BY "group";
""", conn)

,group,total_users,conversions,conversion_rate
0,control,144226,17349,0.1203
1,treatment,144314,17134,0.1187


In [ ]:
data = pd.read_sql("""
SELECT
    "group",
    COUNT(*) AS total_users,
    SUM(converted) AS conversions
FROM ab_data
GROUP BY "group";
""", conn)

data

,group,total_users,conversions
0,control,144226,17349
1,treatment,144314,17134


In [ ]:
control_conv = data.loc[data["group"]=="control", "conversions"].values[0]
control_total = data.loc[data["group"]=="control", "total_users"].values[0]

treat_conv = data.loc[data["group"]=="treatment", "conversions"].values[0]
treat_total = data.loc[data["group"]=="treatment", "total_users"].values[0]

contingency = [
    [control_conv, control_total - control_conv],
    [treat_conv, treat_total - treat_conv]
]

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency)

p

np.float64(0.1975727320140003)

In [ ]:
control = df[df["group"]=="control"]["converted"]
treatment = df[df["group"]=="treatment"]["converted"]

In [ ]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(treatment, control)

p_value

np.float64(0.1955849303673083)

In [ ]:
import numpy as np

cr_control = control.mean()
cr_treatment = treatment.mean()

n_control = len(control)
n_treatment = len(treatment)

In [ ]:
se = np.sqrt(
    (cr_control*(1-cr_control)/n_control) +
    (cr_treatment*(1-cr_treatment)/n_treatment)
)

In [ ]:
diff = cr_treatment - cr_control

lower = diff - 1.96*se
upper = diff + 1.96*se

(lower, upper)

(np.float64(-0.00393041058399703), np.float64(0.0008040950050655538))

In [ ]:
df.to_csv("clean_ab_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("clean_ab_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>